# SatDiff — Colab training

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Nothing is downloaded to your local machine. Data lands in `/content` (wiped each session, ~1 min to re-fetch). Checkpoints go to Drive so a disconnect costs one epoch, not the run.

If Colab disconnects: re-run every cell. Cell 5 resumes automatically.

In [ ]:
# 1. GPU check — stop here if this says False
import torch
print("cuda:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# 2. Drive — checkpoints survive here, /content does not
from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = '/content/drive/MyDrive/satdiff/checkpoints'
import os; os.makedirs(CKPT_DIR, exist_ok=True)
print("checkpoints ->", CKPT_DIR)

In [ ]:
# 3. Code + deps
REPO = 'https://github.com/DrKingSchultz69/satdiff-v2.git'  # <-- your repo URL

%cd /content
![ -d satdiff-v2 ] && (cd satdiff-v2 && git pull) || git clone $REPO
%cd /content/satdiff-v2
!pip install -q -r requirements.txt

import os, sys
os.environ['PYTHONPATH'] = '/content/satdiff-v2/src'
sys.path.insert(0, '/content/satdiff-v2/src')

In [ ]:
# 4. Data — 94 MB, ~1 min. Downloads into /content, not your machine.
!python scripts/download_data.py
!python scripts/make_splits.py

In [ ]:
# 5. Train. --resume makes re-running this cell safe after a disconnect.
!python -m satdiff.train --config configs/v1.yaml --checkpoint-dir $CKPT_DIR --resume

In [ ]:
# 6. Latest fixed-seed grid — one row per class, 4 seeds.
# All four look identical in a row = mode collapse, no matter what KID says.
import glob
from IPython.display import Image, display
grids = sorted(glob.glob('results/grids/*.png'))
print(grids[-1] if grids else 'none yet — first grid appears at epoch 5')
if grids: display(Image(grids[-1]))

In [ ]:
# 7. Eval: KID + CAS. Run this after ~epoch 20.
# CAS < 40% = conditioning is not learning. Stop and fix before burning more GPU.
!python -m satdiff.eval --config configs/v1.yaml --split val --checkpoint-dir $CKPT_DIR